# Ipotesi 1 — LLaVA-Video 7B: il modello usa davvero il video?

Sulle 50 domande di `video_nutrition_estimation` si confrontano le risposte del modello
in tre condizioni. Domanda, opzioni e prompt sono sempre identici; cambia solo l'input visivo.

| Condizione | Input visivo | In questo notebook |
|---|---|---|
| `C0_video` | il video vero (32 frame) | **riusato** dal baseline, si rifanno solo 5 domande di controllo |
| `C1_testo` | nessuno: solo domanda e opzioni | run completo |
| `C3_scambio` | il video di un'altra ricetta, in cui nessuna opzione compare | run completo |

**Scaletta:** 1. ambiente · 2. percorsi e moduli · 3. controllo dei dati · 4. funzioni di
inferenza · 5. prova rapida · 6. run completi · 7. primo sguardo ai numeri

**Prima di iniziare** (vedi `README.md` in questa cartella): `scripts/sync_drive.sh` deve aver copiato
`src/` e `data/` su Drive, con il vecchio risultato di C0 copiato in `results/ipotesi_video/` come
`{modello}_C0_video_video_nutrition_estimation.jsonl`.

## 1. Ambiente (Colab L4)

**Runtime > Cambia tipo di runtime > GPU L4**, in una sessione **nuova**: questo notebook
installa `transformers==4.37.2`, incompatibile con VideoLLaMA 2 (4.40.0).

Identico ai notebook del baseline: stesse versioni, stesso checkpoint. Cambiano solo
installazione e memoria: repo fissato a un commit, `peft` non installato, controlli su
versioni e GPU, e logits calcolati solo sull'ultima posizione (vedi la cella del modello):
senza, su L4 il prefill con 32 fotogrammi va in out of memory.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Runtime: Runtime > Cambia tipo di runtime > GPU L4, sessione NUOVA
!nvidia-smi
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della VRAM
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"                    # evita il timeout del secret HF_TOKEN di Colab
os.environ.pop("HF_TOKEN", None)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Nessuna GPU: Runtime > Cambia tipo di runtime > L4"
print("GPU:", gpu, "| Python:", sys.version.split()[0])
if "L4" not in gpu:
    print("ATTENZIONE: il notebook e' tarato su L4. Un'altra GPU va bene se ha almeno 24 GB.")


In [ ]:
import os, sys

# transformers 4.37.2: l'ultima compatibile con il codice di LLaVA-NeXT (come il baseline).
# Porta con se' tokenizers 0.15.2 e huggingface_hub < 1.0 (wheel anche per Python 3.13).
# accelerate serve per device_map="auto". peft non serve in inferenza e non si installa.
# torch resta quello di Colab, come nel baseline.
# Gli avvisi rossi di pip su gradio, sentence-transformers o peft sono attesi:
# sono pacchetti di Colab che questo notebook non importa.
%pip install -q "transformers==4.37.2" "accelerate>=0.29.1,<2" av timm einops sentencepiece

# Repo ufficiale LLaVA-NeXT, fissato a un commit: se il repo cambia, il codice resta quello del baseline
LLAVA_DIR = "/content/LLaVA-NeXT"
LLAVA_COMMIT = "bce12e479bc4dfee2b9c50c88137b01ff51bd483"   # ultimo commit del repo (giugno 2026)
%cd /content
if not os.path.exists(LLAVA_DIR):
    !git clone -q https://github.com/LLaVA-VL/LLaVA-NeXT.git {LLAVA_DIR}
!git -C {LLAVA_DIR} checkout -q {LLAVA_COMMIT}
!git -C {LLAVA_DIR} log -1 --format="LLaVA-NeXT: commit %h del %cd"
%cd /content/LLaVA-NeXT

# installo il pacchetto llava senza dipendenze (non deve toccare transformers)
%pip install -q -e ".[train]" --no-deps
if LLAVA_DIR not in sys.path:
    sys.path.insert(0, LLAVA_DIR)

from importlib.metadata import version
_vers = {k: version(k) for k in ["torch", "transformers", "tokenizers", "accelerate", "huggingface_hub"]}
print(_vers)
assert _vers["transformers"] == "4.37.2", f"transformers {_vers['transformers']}: controllare l'output di pip sopra"

In [ ]:
import sys, torch, transformers
# Se nella sessione era gia' stato importato un altro transformers (es. il notebook VideoLLaMA 2),
# in memoria c'e' ancora quello: serve una sessione nuova.
assert transformers.__version__ == "4.37.2", (
    f"transformers {transformers.__version__} in memoria: Runtime > Disconnetti ed elimina runtime, poi da capo")
assert torch.cuda.is_available(), "CUDA non disponibile: il runtime non ha la GPU"

if "/content/LLaVA-NeXT" not in sys.path:
    sys.path.insert(0, "/content/LLaVA-NeXT")

# Loader ufficiale lmms-lab: carica pesi, tokenizer e image_processor
# con le chiavi corrette per LLaVA-Video-7B-Qwen2
from llava.model.builder import load_pretrained_model

model_path = "lmms-lab/LLaVA-Video-7B-Qwen2"
model_name = "llava_qwen"   # nome architettura richiesto dal builder

# carico il modello (circa 16 GB da scaricare, piu' la vision tower SigLIP)
tokenizer, model, image_processor, max_length = load_pretrained_model(
    model_path,
    None,           # model_base: None per modelli stand-alone
    model_name,
    torch_dtype="bfloat16",
    device_map="auto",
    attn_implementation="sdpa",
)

# mettiamo in modalita' evaluation
model.eval()

# Tutto il modello deve stare sulla GPU: con device_map="auto", se la memoria non basta,
# accelerate sposta dei layer sulla CPU senza errori ma rende l'inferenza lentissima.
fuori = {str(p.device) for p in model.parameters() if p.device.type != "cuda"}
assert not fuori, f"parte del modello e' su {fuori} (offload): serve piu' VRAM libera"
device = next(model.parameters()).device

# L4: logits solo per l'ultima posizione.
# transformers 4.37 nel prefill calcola i logits su TUTTA la sequenza (circa 6.900 token con
# 32 fotogrammi) e li converte in float32: 6.900 x 152.064 voci = circa 5,9 GiB in piu' oltre
# ai 15 GiB di pesi. Su A100 c'era spazio, su L4 (22 GiB utili) e' out of memory.
# Con decoding greedy si usa solo la riga dell'ultima posizione, quindi l'hook passa al lm_head
# solo quella. Gli hidden state sono gli stessi; puo' cambiare solo l'arrotondamento del
# prodotto finale, che conta solo se due lettere sono praticamente alla pari.
if globals().get("_hook_lm_head") is not None:
    _hook_lm_head.remove()
_hook_lm_head = model.lm_head.register_forward_pre_hook(lambda modulo, args: (args[0][:, -1:, :],))

print(f"LLaVA-Video-7B-Qwen2 caricato su {device}")
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1024**3:.1f} GiB "
      f"su {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

## 2. Percorsi e moduli

I moduli si importano da `src/hdepic/` su Drive: niente `%%writefile`.
`hdepic.eval` e' la stessa versione usata per il baseline.

In [ ]:
import os, sys, glob, json, importlib

BASE    = "/content/drive/MyDrive/Progetto Deep Learning 1"   # cartella del progetto su Drive
SRC      = os.path.join(BASE, "src")                            # codice, copiato su Drive da scripts/sync_drive.sh
DERIVATI = os.path.join(BASE, "data", "derived", "ipotesi_video")
ANN_ORIG = os.path.join(BASE, "data", "hd-epic")                # annotazioni originali del benchmark
ANN_SWAP = os.path.join(DERIVATI, "annotazioni_swap")           # annotazioni con il video scambiato (C3)
OUT_DIR  = os.path.join(BASE, "results", "ipotesi_video")
os.makedirs(OUT_DIR, exist_ok=True)
MODELLO  = "llavavideo"

sys.path.insert(0, SRC)
from hdepic import eval as H
from hdepic import ablazione as A
importlib.reload(H); importlib.reload(A)
print(f"hdepic.eval {H.VERSION} | ablazione {A.VERSION}")

with open(os.path.join(DERIVATI, "swap_map.json")) as f:
    SWAP_MAP = json.load(f)

VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
print("cartelle video:", [os.path.basename(d) for d in VIDEO_ROOTS])
print(f"video campionati indicizzati: {len(VIDEO_INDEX['sampled'])}")

c0 = A.percorso(OUT_DIR, MODELLO, "C0_video")
print(f"\nC0 (video vero): {'trovato' if os.path.exists(c0) else 'MANCANTE'} -> {os.path.basename(c0)}")

## 3. Controllo dei dati

Prima di girare si verifica che:
- le annotazioni scambiate differiscano da quelle originali **solo** per l'id del video;
- tutti i video servano davvero: quelli originali (stesse domande di C0) e quelli di scambio.

Le attese sono **50 domande utilizzabili** in entrambe le tabelle. Se ne mancano, il confronto
con C0 resta valido ma va fatto sulle sole domande comuni (lo gestisce `03_analisi`).

In [ ]:
READABLE = {}   # cache: aprire i video da Drive e' lento, si controlla una volta sola

A.controlla_annotazioni_scambiate(ANN_ORIG, ANN_SWAP)
for nome, ann_dir in [("annotazioni originali (C1, controllo C0)", ANN_ORIG),
                      ("annotazioni scambiate (C3)", ANN_SWAP)]:
    cov, dettagli = H.preflight(ann_dir, VIDEO_INDEX, tasks=[A.TASK], cache=READABLE)
    print(f"\n{nome}")
    print(cov.to_string(index=False))
    rotti = sorted({v for ids in dettagli[A.TASK].values() for v in ids})
    if rotti:
        print("  file mancanti o non decodificabili:", rotti)

## 4. Funzioni di inferenza

Rispetto al baseline la funzione e' solo divisa in due, senza cambiare nulla del calcolo:
- **`_genera`** costruisce il prompt nel template `qwen_1_5` e genera la risposta (greedy);
- **`infer_video`** prepara i 32 fotogrammi e mette il token immagine davanti al prompt,
  esattamente come `llava_infer` del baseline;
- **`infer_testo`** passa lo stesso prompt senza token immagine e senza `images`: in quel caso
  `generate` di LLaVA-NeXT usa solo gli embedding del testo.

In [ ]:
import av
import numpy as np
from PIL import Image
from llava.conversation import conv_templates
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.mm_utils import tokenizer_image_token

N_FRAMES = 32
MAX_NEW_TOKENS = 32

def sample_video_frames(video_path, n=N_FRAMES):
    """n fotogrammi uniformi con PyAV (come il codebase ufficiale di LLaVA-Video)."""
    container = av.open(video_path)
    total = container.streams.video[0].frames
    if total == 0:                                  # metadato assente: li conto
        total = sum(1 for _ in container.decode(video=0))
        container.close()
        container = av.open(video_path)
    voluti = set(np.linspace(0, total - 1, min(n, total), dtype=int).tolist())
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i in voluti:
            frames.append(frame.to_ndarray(format="rgb24"))
        if len(frames) == len(voluti):
            break
    container.close()
    return frames

def _immagini(visual, n=N_FRAMES):
    """path mp4 o lista di path -> lista di PIL.Image (liste lunghe sottocampionate a n)."""
    if isinstance(visual, list):
        paths = visual
        if len(paths) > n:
            sel = np.linspace(0, len(paths) - 1, n, dtype=int)
            paths = [paths[i] for i in sel]
        return [Image.open(p).convert("RGB") for p in paths]
    return [Image.fromarray(f) for f in sample_video_frames(visual, n)]

def _genera(messaggio, video_tensor=None):
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], messaggio)
    conv.append_message(conv.roles[1], None)
    input_ids = tokenizer_image_token(conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX,
                                      return_tensors="pt").unsqueeze(0).to(device)
    visivo = {} if video_tensor is None else \
             dict(images=[video_tensor], image_sizes=None, modalities=["video"])
    with torch.no_grad():
        out = model.generate(input_ids, **visivo, do_sample=False, temperature=None,
                             max_new_tokens=MAX_NEW_TOKENS, use_cache=True)
    testo = tokenizer.decode(out[0], skip_special_tokens=False)
    marker = "<|im_start|>assistant\n"
    if marker in testo:
        return testo.split(marker)[-1].replace("<|im_end|>", "").strip()
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def infer_video(visual, prompt):
    """visual: path di un mp4 oppure lista di path di immagini."""
    video_tensor = image_processor.preprocess(_immagini(visual), return_tensors="pt")["pixel_values"]
    video_tensor = video_tensor.to(device=device, dtype=torch.bfloat16)
    return _genera(f"{DEFAULT_IMAGE_TOKEN}\n{prompt}", video_tensor)

def infer_testo(prompt):
    """Solo testo: nessun token immagine, nessun input visivo."""
    return _genera(prompt)

# condizione -> (funzione di inferenza, annotazioni da usare)
INFER = {"C1_testo": A.solo_testo(infer_testo), "C3_scambio": infer_video}
ANN   = {"C1_testo": ANN_ORIG,                  "C3_scambio": ANN_SWAP}
print(f"pronto - {N_FRAMES} fotogrammi per domanda col video, decoding greedy")

## 5. Prova rapida

Tre domande per condizione, scritte in file `*_prova_*` separati dai run completi.

**Cosa controllare a occhio:**
- **C1** le risposte sono lettere (`A`–`E`) e `parse_method` e' `leading_letter`;
- **C3** il video indicato e' diverso dall'originale tra parentesi;
- nessuna riga con `ERRORE`.

In [ ]:
prova = {c: A.esegui(MODELLO, c, INFER[c], ANN[c], VIDEO_INDEX, OUT_DIR,
                     limit=3, prova=True, readable_cache=READABLE)
         for c in ["C1_testo", "C3_scambio"]}
A.mostra_prova(prova, SWAP_MAP)

### Controllo dell'ambiente su C0

C0 non viene rifatto: si riusano i risultati del baseline. Per essere sicuri che questa
sessione Colab si comporti come quella del baseline, si rifanno **5 domande col video vero**.
Con il decoding greedy le lettere devono essere **identiche** a quelle salvate in C0.

In [ ]:
ctrl = A.esegui(MODELLO, "C0_controllo", infer_video, ANN_ORIG, VIDEO_INDEX, OUT_DIR,
                limit=5, readable_cache=READABLE)
_ = A.confronta_controllo(A.percorso(OUT_DIR, MODELLO, "C0_video"),
                          A.percorso(OUT_DIR, MODELLO, "C0_controllo"))

## 6. Run completi

Ogni risposta viene scritta subito sul `.jsonl`: se Colab si disconnette, rilanciare la
cella riprende dalla domanda successiva.

In [ ]:
risultati = {c: A.esegui(MODELLO, c, INFER[c], ANN[c], VIDEO_INDEX, OUT_DIR,
                         readable_cache=READABLE)
             for c in ["C1_testo", "C3_scambio"]}

## 7. Primo sguardo ai numeri

Solo per questo modello. L'analisi completa, con entrambi i modelli, i grafici e gli esempi,
e' in `03_analisi.ipynb`.

In [ ]:
df = A.controlla_allineamento(A.carica_risultati(OUT_DIR, [MODELLO]))
tab = A.tabella_principale(df)
display(tab)
_ = A.verdetto(tab)